# Project: Digital Restoration of Motion Blur using Comparative Filtering
**Author:** Lorenzo Pasini


**Objective:** This notebook explores the implementation and application of various deconvolution filters to restore images degraded by motion blur. The study focuses on analyzing different filtering techniques and evaluating their performance through both visual results and quantitative metrics.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage import data, color
from skimage.util import img_as_float
from skimage.metrics import (
    peak_signal_noise_ratio as psnr,
    structural_similarity as ssim
)

plt.rcParams['figure.figsize'] = (15, 10)
plt.rcParams['image.cmap'] = 'gray'

## 1) Motion Blur model

The image degradation process caused by linear motion during the exposure time $T$ can be described in the frequency domain by the transfer function $H(u, v)$.

Starting from the general temporal integration model:
$$H(u, v) = \int_{0}^{T} e^{-j2\pi [u x_0(t) + v y_0(t)]} \, dt$$

Assuming **uniform linear motion**, we define the displacement components as:
* $x_0(t) = \frac{at}{T}$
* $y_0(t) = \frac{bt}{T}$
where $a$ and $b$ represent the total displacement along the $x$ and $y$ axes at the end of the exposure interval $T$. Substituting these expressions into the integral, we obtain:

$$H(u, v) = \int_{0}^{T} e^{-j\frac{2\pi t}{T} (ua + vb)} \, dt$$

Solving the integral of the exponential function:

$$H(u, v) = \left[ \frac{e^{-j\frac{2\pi t}{T} (ua + vb)}}{-j\frac{2\pi}{T} (ua + vb)} \right]_{0}^{T}$$

Evaluating at the integration limits $[0, T]$:

$$H(u, v) = \frac{e^{-j2\pi (ua + vb)} - 1}{-j\frac{2\pi}{T} (ua + vb)}$$

To transform this expression into the sinc function form, we use Euler's identity by factoring out the exponential term $e^{-j\pi(ua+vb)}$:

$$H(u, v) = \frac{e^{-j\pi (ua + vb)} \left( e^{-j\pi (ua + vb)} - e^{j\pi (ua + vb)} \right)}{-j\frac{2\pi}{T} (ua + vb)}$$

Using the definition of the sine function $\sin(\theta) = \frac{e^{j\theta} - e^{-j\theta}}{2j}$, the expression becomes:

$$H(u, v) = \frac{e^{-j\pi (ua + vb)} \cdot [-2j \sin(\pi(ua + vb))]}{-j\frac{2\pi}{T} (ua + vb)}$$

Simplifying the constant terms and rearranging, we reach the final form:

$$H(u, v) = \frac{T}{\pi(ua + vb)} \sin(\pi(ua + vb)) e^{-j\pi(ua + vb)}$$

By using the normalized sampling function $\text{sinc}(x) = \frac{\sin(\pi x)}{\pi x}$, the filter can be synthesized as:

$$H(u, v) = T \cdot \text{sinc}(ua + vb) \cdot e^{-j\pi(ua + vb)}$$

In [ ]:
def get_motion_blur_transfer_function(shape, a=0.1, b=0.1, T=1.0):
    """
    Computes the frequency domain transfer function for linear motion blur.

    :param shape: Tuple representing (rows, cols) of the image.
    :param a: Displacement rate in the horizontal direction.
    :param b: Displacement rate in the vertical direction.
    :param T: Exposure duration.
    :return: 2D complex array of the blur filter H(u, v).
    """
    # Getting the dimension of the image
    M, N = shape

    # Create frequency coordinates centered at (0,0)
    u = np.arange(-M // 2, M // 2) # Discrete interval of M values
    v = np.arange(-N // 2, N // 2) # Discrete interval of N values
    V, U = np.meshgrid(v, u) # Creating a meshgrid of the two vectors (Numpy indexing convention)

    # Compute common argument for the sinc and exponent components
    arg = U * a + V * b

    # H(u,v) = T * sinc(arg) * exp(-j * π * arg)
    H_transfer = T * np.sinc(arg) * np.exp(-1j * np.pi * arg)

    # Returning the frequency filter
    return H_transfer

def get_motion_blurred_image(image, a, b, T):
    """
    Applies motion blur to an image using the frequency domain model.

    :param image: Input spatial domain image.
    :param a: Horizontal motion parameter.
    :param b: Vertical motion parameter.
    :param T: Exposure parameter.
    :return: Tuple of (blurred_image, H_transfer).
    """
    # Move image to frequency domain and center the low frequencies
    F = np.fft.fftshift(np.fft.fft2(image))

    # Generate the degradation filter
    H_transfer = get_motion_blur_transfer_function(image.shape, a, b, T)

    # Apply degradation: G(u,v) = H(u,v) * F(u,v)
    G = H_transfer * F

    # Return to spatial domain via Inverse FFT
    G_shift = np.fft.ifftshift(G)
    blurred_image = np.real(np.fft.ifft2(G_shift))

    # Return the real part
    return blurred_image, H_transfer

## 2) Filter implementations
In the frequency domain, an image with noise and distortion is modeled as:
$$G(u,v) = F(u,v)H(u,v) + N(u,v)$$
where:
* $G(u, v)$ is the Discrete Fourier Transform (DFT) of the degraded image.
* $F(u, v)$ is the DFT of the original, undegraded image.
* $H(u, v)$ is the Optical Transfer Function (OTF) representing the degradation.
* $N(u, v)$ is the DFT of the additive noise.

Or in the spatial domain:
$$g(x,y) = f(x,y) * h(x,y) + \eta(x,y)$$
where:
* $g(x, y)$ is the observed degraded image.
* $f(x, y)$ is the original scene.
* $h(x, y)$ is the Point Spread Function (PSF).
* $\eta(x, y)$ is the additive noise term.
* $*$ denotes the 2D convolution operator.

### Inverse Filter

The inverse filter is the simplest approach to restoration. In the frequency domain, we attempt to recover the original image $F(u,v)$ by dividing the degraded image $G(u,v)$ by the degradation function $H(u,v)$:

$$\hat{F}(u,v) = \frac{G(u,v)}{H(u,v)} = \frac{F(u,v)H(u,v) + N(u,v)}{H(u,v)} = F(u,v) + \frac{N(u,v)}{H(u,v)}$$

where:
* $\hat{F}(u,v)$ is the estimate of the original image.
* $N(u,v)/H(u,v)$ represents the noise term amplified by the inverse of the degradation function.

This approach reveals a critical vulnerability: if the degradation function $H(u,v)$ has tiny values or zeros (which is common in motion blur or defocus), the term $1/H(u,v)$ becomes extremely large. This causes the additive noise $N(u,v)$ to be amplified significantly, often completely overwhelming the signal $F(u,v)$ in the restored image. For this reason it is implemented using a threshold.

In [ ]:
def apply_inverse_filter(degraded_img, H_transfer, threshold=0.1):
    """
    Restores an image using an inverse filter with a frequency threshold.

    :param degraded_img: The blurred/noisy input image.
    :param H_transfer: The degradation transfer function H(u, v).
    :param threshold: Minimum size of H to allow inversion (prevents noise explosion).
    :return: Restored image clipped to valid intensity range [0, 1].
    """
    # Transform degraded image to frequency domain
    G = np.fft.fft2(degraded_img)

    # Initialize inverse filter array
    H_inv = np.zeros_like(H_transfer, dtype=complex)

    # Invert H only where magnitude exceeds threshold to mitigate noise amplification
    mask = np.abs(H_transfer) > threshold
    H_inv[mask] = 1.0 / H_transfer[mask]

    # Frequency domain restoration: F_hat(u,v) = G(u,v) * H_inv(u,v)
    F_hat = G * H_inv

    # Return to spatial domain and clip pixel values to [0, 1]
    restored_img = np.real(np.fft.ifft2(F_hat))

    # Clip pixel values to [0, 1]
    return np.clip(restored_img, 0, 1)

### Wiener Filter

The Wiener filter is defined in the frequency domain as:
$$H_w(u,v) = \frac{H^*(u,v)}{|H(u,v)|^2 + \frac{S_{\eta}(u,v)}{S_f(u,v)}}$$
where:
* $H^*(u,v)$ is the complex conjugate of the degradation function.
* $S_{\eta}(u,v) = |N(u,v)|^2$ is the power spectrum of the noise.
* $S_f(u,v) = |F(u,v)|^2$ is the power spectrum of the original image.

In practical applications, the power spectra $S_{\eta}$ and $S_f$ are rarely known. Consequently, the noise-to-signal power ratio is often approximated by a constant $K$:
$$K \approx \frac{S_{\eta}(u,v)}{S_f(u,v)}$$

This constant $K$ is related to the **Signal-to-Noise Ratio (SNR)**, defined as:
$$SNR = \frac{\sum_{u,v} |F(u,v)|^2}{\sum_{u,v} |N(u,v)|^2} \approx \frac{1}{K}$$

In this notebook, we treat $K$ as a tunable hyperparameter. It is important to observe the limiting behavior of the filter:
*   As $K \to 0$ (which implies $SNR \to \infty$), the noise is assumed to be negligible. In this case, the Wiener filter reduces to the **Inverse Filter**:
    $$\lim_{K \to 0} H_w(u,v) = \frac{H^*(u,v)}{|H(u,v)|^2} = \frac{1}{H(u,v)}$$
*   As $K$ increases, the filter becomes more conservative, suppressing noise at the expense of high-frequency details (blurring).

Therefore, the Wiener filter acts as a "regularized" inverse filter, where $K$ prevents the denominator from vanishing at frequencies where $H(u,v)$ is small, effectively avoiding the noise amplification typical of the pure inverse filtering approach.

In [ ]:
def apply_wiener_filter(degraded_img, H_transfer, K=0.01):
    """
    Restores an image using the Wiener filter (Minimum Mean Square Error).

    :param degraded_img: The blurred/noisy input image.
    :param H_transfer: The degradation transfer function H(u, v).
    :param K: Constant representing the noise-to-signal power ratio.
    :return: Restored image clipped to valid intensity range [0, 1].
    """
    # Transform degraded image to frequency domain
    G = np.fft.fft2(degraded_img)

    # Compute filter components: |H(u,v)|^2 and conjugate H*(u,v)
    H_conj = np.conj(H_transfer)
    H_square_module = np.abs(H_transfer) ** 2

    # Wiener Filter formula: H_w = H* / (|H|^2 + K)
    H_w = H_conj / (H_square_module + K)

    # Apply filter and return to spatial domain
    restored_img = np.real(np.fft.ifft2(G * H_w))

    # Clip pixel values to [0, 1]
    return np.clip(restored_img, 0, 1)

### Constrained Least Squares Filtering (CLSF)

The **Constrained Least Squares Filtering (CLSF)** is designed to overcome the sensitivity of the Inverse Filter to noise without requiring the explicit knowledge about the power spectra of the noise and the original image.

#### Problem Formulation
Given the degradation model $g = Hf + \eta$ (another way of writing $G(u,v) = F(u,v)H(u,v) + N(u,v)$), where $\eta$ is additive noise, the CLSF approach treats restoration as a constrained optimization problem. The goal is to minimize a measure of **smoothness**, specifically the norm relating to the Laplacian of the image—subject to a constraint on the fidelity to the observed data:

1.  **Minimize:** $\sum_{x,y} | \nabla^2 f(x,y) |^2$
2.  **Subject to:** $\| g - H\hat{f} \|^2 = \| \eta \|^2$

This formulation assumes that the noise norm $\|\eta\|^2$ (or its variance) is roughly known, and we seek the smoothest possible image that remains consistent with the degraded observation $g$.

#### The Restoration Filter
By solving this optimization using Lagrange multipliers, we get the restoration filter in the frequency domain:

$$\hat{F}(u,v) = \left[ \frac{H^*(u,v)}{|H(u,v)|^2 + \gamma |P(u,v)|^2} \right] G(u,v)$$

Where:
*   **$H^*(u,v)$** is the complex conjugate of the degradation function.
*   **$|P(u,v)|^2$** is the Fourier Transform of the Laplacian operator $p(x,y)$. The Laplacian kernel $p(x,y)$ is typically defined as:
    $$p(x,y) = \begin{bmatrix} 0 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 0 \end{bmatrix}$$
*   **$\gamma$** is a regularization parameter (Lagrange multiplier) that balances the trade-off between smoothness and data fidelity.

#### Advantages
Unlike the Wiener filter, which requires the ratio $S_{\eta}/S_f$, CLSF only requires the knowledge of $H(u,v)$ and the definition of a high-pass operator $P(u,v)$. By tuning $\gamma$, we can effectively suppress noise amplification at high frequencies while preserving the structural integrity of the image.

In [ ]:
def apply_clsf_filter(degraded_img, H_transfer, gamma=0.05, noise_freq=None):
    """
    Restores an image using the Constrained Least Squares Filter (CLSF).

    :param degraded_img: The blurred/noisy input image.
    :param H_transfer: The degradation transfer function H(u, v).
    :param gamma: Regularization parameter controlling the smoothness (is set to 0, optimum will be calculated automatically).
    :return: Restored image clipped to valid intensity range [0, 1] and the optimal gamma value (if calculated, else the gamma parameter is returned).
    :param noise_freq: Noise frequency (required if gamma is set to 0).
    """
    # Transform degraded image to frequency domain
    G = np.fft.fft2(degraded_img)

    # Getting the image size
    M, N = G.shape

    # Define the 3x3 Laplacian operator for smoothness constraint
    p = np.array([[ 0, -1,   0],
                  [-1,   4, -1],
                  [ 0, -1,   0]])

    # Pad the Laplacian to match image dimensions
    P_padded = np.zeros((M, N))
    P_padded[:3, :3] = p

    # Center the operator to avoid phase shifts in the frequency domain
    P_padded = np.roll(P_padded, -1, axis=0)
    P_padded = np.roll(P_padded, -1, axis=1)

    # Compute the Fourier Transform of the Laplacian
    P = np.fft.fft2(P_padded)

    # Compute filter components
    H_conj = np.conj(H_transfer)
    H_square_module = np.abs(H_transfer) ** 2
    P_square_module = np.abs(P)**2

    # Gamma retrival if parameter gamma is set to 0
    if gamma == 0:
        if noise_freq is None:
            raise ValueError("noise_freq must be provided to find optimal gamma.")

        gamma_calc = find_optimal_gamma(G, H_transfer, P, M, N, noise_freq, accuracy_value=1e-5, max_iteration=100)
    else:
        gamma_calc = gamma


    # CLSF formula: H_c = H* / (|H|^2 + gamma * |P|^2)
    # Added epsilon (1e-12) to prevent division by zero
    H_c = H_conj / (H_square_module + gamma_calc * P_square_module + 1e-12)

    # Apply filter and return to spatial domain
    restored_img = np.real(np.fft.ifft2(G * H_c))

    # Clip pixel values to [0, 1]
    return np.clip(restored_img, 0, 1), gamma_calc

### Automatic Selection of $\gamma$

In the CLSF framework, the optimal $\gamma$ is the one that satisfies the constraint $\| \mathbf{g} - \mathbf{H}\mathbf{\hat{f}} \|^2 = \| \boldsymbol{\eta} \|^2$. We define the **residual vector** $\mathbf{r}$ as:
$$\mathbf{r} = \mathbf{g} – \mathbf{H}\mathbf{\hat{f}}$$

#### 1. Residual in the Frequency Domain
Using the DFT properties, the residual in the frequency domain is given by:
$$R(u,v) = G(u,v) - H(u,v)\hat{F}(u,v)$$

Substituting the CLSF expression for $\hat{F}(u,v)$:
$$R(u,v) = G(u,v) \left[ 1 – \frac{|H(u,v)|^2}{|H(u,v)|^2 + \gamma |P(u,v)|^2} \right]$$

#### 2. Noise Estimation
The norm of the noise $\| \boldsymbol{\eta} \|^2$ can be estimated if we assume the noise statistics are known or can be measured from a constant area of the image. The variance $\sigma_\eta^2$ and the mean $m_\eta$ are defined as:
$$m_\eta = \frac{1}{MN} \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} \eta(x,y)$$
$$\sigma_\eta^2 = \frac{1}{MN} \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} (\eta(x,y) - m_\eta)^2$$

From these, the squared norm of the noise vector is calculated as:
$$\|\boldsymbol{\eta}\|^2 = MN(\sigma_\eta^2 + m_\eta^2)$$

#### 3. Iterative Optimization
The residual power $\phi(\gamma) = \|\mathbf{r}\|^2$ is a monotonically increasing function of $\gamma$. By applying Parseval's theorem, we calculate this power in the frequency domain:
$$\|\mathbf{r}\|^2 = \frac{1}{MN} \sum_{u=0}^{M-1} \sum_{v=0}^{N-1} |R(u,v)|^2$$

The algorithm seeks the value of $\gamma$ such that:
$$\phi(\gamma) = \|\boldsymbol{\eta}\|^2 \pm a$$
where $a$ is a small precision tolerance. In practice, this is solved using numerical methods like the **Newton-Raphson** method or a simple **bisection search** because of the monotonic nature of $\phi(\gamma)$.

In [ ]:
def find_optimal_gamma(G, H_transfer, P, M, N, noise_freq, gamma_init = 0.5, accuracy_value=1e-5, max_iteration=100):
    """
    Iteratively finds the optimal gamma for CLSF using the bisection method.

    :param G: Degraded image in the frequency domain.
    :param H_transfer: Degradation transfer function H(u, v).
    :param P: Laplacian operator in the frequency domain.
    :param M: Image height.
    :param N: Image width.
    :param noise_freq: Frequency of the noise component.
    :param gamma_init: Initial regularization value.
    :param accuracy_value: Tolerance for convergence.
    :param max_iteration: Maximum number of bisection steps.
    :return: Optimized gamma value.
    """
    # Beginning parameters
    gamma = gamma_init
    low = 0.0
    high = 1e9

    # Pre-calculate squared magnitudes
    H_abs_square = np.abs(H_transfer) ** 2
    P_abs_square = np.abs(P)**2

    # Estimate noise characteristics from the noise component eta
    eta = np.fft.ifft2(noise_freq).real
    mean_noise = float(np.mean(eta))
    noise_var = np.var(eta)

    # Define target residual norm: ||eta||^2 = MN * (sigma^2 + mean^2)
    target_norm_sq = M * N * (noise_var + mean_noise**2)

    # Computation by iteration
    for j_for in range(max_iteration):
        # Compute the residual R(u,v) = G(u,v) * [gamma*|P|^2 / (|H|^2 + gamma*|P|^2)]
        R = G * ( (gamma * P_abs_square) / (H_abs_square + gamma * P_abs_square + 1e-12))

        # Compute squared norm of the residual in the spatial domain (via Parseval)
        R_norm_sq = np.vdot(R, R).real / (M * N)

        # Check for convergence
        if np.abs(R_norm_sq - target_norm_sq) < accuracy_value * target_norm_sq:
            break

        # Adjust gamma using bisection logic
        if R_norm_sq < target_norm_sq:
            low = gamma # Residue too small -> larger gamma
        else:
            high = gamma # Residue too large -> smaller gamma

        gamma = (low + high) / 2

    return gamma

## 3) Testing Different Filters
We now analyze the filtering performance on three test images degraded by motion blur and additive Gaussian noise with different intensities. The goal is to evaluate how the different restoration techniques behave under realistic degradation conditions and compare their reconstruction quality.

### 3.1) Parameters
We first define the parameters required for the image degradation process and for the restoration filters used in the experiments.

In [ ]:
# Function to safely convert to gray only if the image is RGB
def to_gray(img):
    if img.ndim == 3:
        return color.rgb2gray(img)
    return img

# --- Dataset Setup ---
image_names = ['Astronaut', 'Coffee', 'Cat']
original_images = [
    img_as_float(to_gray(data.astronaut())),
    img_as_float(to_gray(data.coffee())),
    img_as_float(to_gray(data.cat()))
]

# --- Global Configuration ---
np.random.seed(42)
num_imgs = len(original_images)
num_settings = 5

# --- Degradation Parameters (5 Diverse Scenarios) ---
# Scenario: [V-Blur, H-Blur, Diag-Blur, Strong-Diag-Blur, High-Motion-Blur]
a_params = [0.10, 0.00, 0.05, 0.08, 0.02]  # Vertical motion component
b_params = [0.00, 0.10, 0.05, 0.08, 0.02]  # Horizontal motion component
T_params = [1.00, 1.00, 0.80, 1.50, 0.50]  # Time/Duration of exposure

# --- Noise & Restoration Parameters ---
noise_variance = [0.0005, 0.0, 0.0004, 0.003, 0.003]
noise_presence = [True, False, True, True, True]

# --- Filter parameters ---
inverse_threshold = [0.01, 0.03, 0.10, 0.15, 0.05]
wiener_K = [0.05, 0.04, 0.02, 0.01, 0.01]
clsf_gamma = [0, 0, 0, 0, 0] # 0 to be automatic!

### 3.2) Visualizing the original images
We now display the original images before degradation, which will serve as a reference for evaluating the quality of the restored results.

In [ ]:
# Printing the original images
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle('Original Images', fontsize=16)

for i in range(len(original_images)):
    axes[i].imshow(original_images[i], cmap='gray')
    axes[i].set_title(f"{image_names[i]}\n")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### 3.3) Making the degraded images
We now generate the degraded test images using the well-known samples astronaut, coffee, cat from the original dataset. Each image is degraded by applying motion blur and additive Gaussian noise (with different intensities) to simulate realistic acquisition imperfections.

In [ ]:
# Each image is blurred using different type of degradation settings
H_filters = [[None for _ in range(num_settings)] for _ in range(num_imgs)]
degraded_images = [[None for _ in range(num_settings)] for _ in range(num_imgs)]
noises_spatial = [[None for _ in range(num_settings)] for _ in range(num_imgs)]

for i in range(int(num_imgs)):
    # Degradation of image i using the settings
    for j in range(int(num_settings)):
        # Blurring image i using setting j
        blurred_float, H = get_motion_blurred_image(
            original_images[i],
            a_params[j],
            b_params[j],
            T_params[j]
        )

        H_filters[i][j] = H # type: ignore

        # Create degraded image
        sigma = np.sqrt(noise_variance[j])
        noise_spatial = np.random.normal(0, sigma, blurred_float.shape)
        noises_spatial[i][j] = noise_spatial # type: ignore

        # Degradation blur + noise
        degraded = blurred_float + (noise_presence[j] * noise_spatial)

        # Clipping and saving degraded images
        degraded_images[i][j] = (np.clip(degraded, 0, 1)) # type: ignore

### 3.4) Visualizing the degraded images
In this section, we display the degraded versions of the images obtained after applying motion blur and additive Gaussian noise. These results will be used as the input for the subsequent restoration step and for comparison with the original images.

In [ ]:
# Plotting 5 images per row
fig, axes = plt.subplots(num_imgs, num_settings, figsize=(20, 12))
fig.suptitle('Degraded Images (Motion Blur + Additive Noise)', fontsize=16)
for i in range(num_imgs):
    for j in range(num_settings):
        axes[i, j].imshow(degraded_images[i][j], cmap='gray')

        if i == 0:
            axes[i, j].set_title(f"Set {j}\n(a={a_params[j]}, b={b_params[j]}, T={T_params[j]})")

        if j == 0:
            axes[i, j].set_ylabel(image_names[i], size='large')

        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.tight_layout()
plt.show()

### 3.5) Applying inverse filter
In this step, we apply the inverse filtering technique to the degraded images to attempt reconstruction of the original content. This method is used as a baseline restoration approach, and its performance will be evaluated against more robust filtering methods.

In [ ]:
# Applying the inverse filter
restored_images_inverse = [[None for _ in range(num_settings)] for _ in range(num_imgs)]
for i in range(num_imgs):
    for j in range(num_settings):
        H_for_filter = np.fft.ifftshift(H_filters[i][j])
        restored = apply_inverse_filter(degraded_images[i][j], H_for_filter, inverse_threshold[j])
        restored_images_inverse[i][j] = restored # type: ignore

In [ ]:
# Plotting Restored Images
fig, axes = plt.subplots(num_imgs, num_settings, figsize=(20, 12))
fig.suptitle('Restored Images using Inverse Filter', fontsize=16)
for i in range(num_imgs):
    for j in range(num_settings):
        axes[i, j].imshow(restored_images_inverse[i][j], cmap='gray')

        if i == 0:
            axes[i, j].set_title(f"Set {j}\n(a={a_params[j]}, b={b_params[j]}, T={T_params[j]})\n"
                                 f"Inverse Threshold: {inverse_threshold[j]}")

        if j == 0:
            axes[i, j].set_ylabel(image_names[i], size='large')

        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.tight_layout()
plt.show()

### 3.6) Applying Wiener filter
In this step, we apply the Wiener filter to the degraded images to perform noise-robust restoration. Unlike inverse filtering, this approach incorporates statistical information about both the degradation and the noise, aiming to produce a more stable reconstruction under realistic conditions.

In [ ]:
# Applying the Wiener filter
restored_images_wiener = [[None for _ in range(num_settings)] for _ in range(num_imgs)]
for i in range(num_imgs):
    for j in range(num_settings):
        H_for_filter = np.fft.ifftshift(H_filters[i][j])
        restored = apply_wiener_filter(degraded_images[i][j], H_for_filter, wiener_K[j])
        restored_images_wiener[i][j] = restored # type: ignore

In [ ]:
# Plotting Restored Images
fig, axes = plt.subplots(num_imgs, num_settings, figsize=(20, 12))
fig.suptitle('Restored Images using Wiener Filter', fontsize=16)
for i in range(num_imgs):
    for j in range(num_settings):
        axes[i, j].imshow(restored_images_wiener[i][j], cmap='gray')

        if i == 0:
            axes[i, j].set_title(f"Set {j}\n(a={a_params[j]}, b={b_params[j]}, T={T_params[j]})\n "
                                 f"K: {wiener_K[j]}")

        if j == 0:
            axes[i, j].set_ylabel(image_names[i], size='large')

        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.tight_layout()
plt.show()

### 3.7) Applying CLSF filter
In this final restoration step, we apply the Constrained Least Squares Filter (CLSF) to the degraded images. This method introduces a regularization term to control noise amplification while still enforcing fidelity to the observed degraded image, typically leading to improved stability and reconstruction quality compared to purely inverse-based approaches.

In [ ]:
# Applying the CLSF filter
restored_images_clsf = [[None for _ in range(num_settings)] for _ in range(num_imgs)]
gammas_c = [[None for _ in range(num_settings)] for _ in range(num_imgs)]
for i in range(num_imgs):
    for j in range(num_settings):
        H_for_filter = np.fft.ifftshift(H_filters[i][j])
        restored, gamma_c = apply_clsf_filter(degraded_images[i][j], H_for_filter, clsf_gamma[j], noise_freq = np.fft.fft2(noises_spatial[i][j]))
        restored_images_clsf[i][j] = restored # type: ignore
        gammas_c[i][j] = gamma_c # type: ignore

In [ ]:
# Plotting Restored Images
fig, axes = plt.subplots(num_imgs, num_settings, figsize=(20, 12))
fig.suptitle('Restored Images using CLSF Filter', fontsize=16)
for i in range(num_imgs):
    for j in range(num_settings):
        axes[i, j].imshow(restored_images_clsf[i][j], cmap='gray')

        if i == 0:
            axes[i, j].set_title(f"Set {j}\n(a={a_params[j]}, b={b_params[j]}, T={T_params[j]})")

        if j == 0:
            axes[i, j].set_ylabel(image_names[i], size='large')

        axes[i, j].set_title(f"gamma={gammas_c[i][j]:.4f}")

        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])

plt.tight_layout()
plt.show()

### 3.8) Filter comparison
We now compare the results obtained using inverse filtering, Wiener filtering, and the Constrained Least Squares Filter (CLSF). The comparison is carried out both visually and, where applicable, through quantitative metrics to assess reconstruction quality, noise suppression, and preservation of image details.

In [ ]:
def style_metrics_table(df, table_title):
    cols_ssim = ['SSIM Inverse', 'SSIM Wiener', 'SSIM CLSF']
    cols_psnr = ['PSNR Inverse', 'PSNR Wiener', 'PSNR CLSF']
    return df.style.background_gradient(
        cmap='RdYlGn', subset=cols_ssim, axis=1
    ).background_gradient(
        cmap='RdYlGn', subset=cols_psnr, axis=1
    ).format(precision=4).set_caption(table_title)

styled_tables = []

for i in range(num_imgs):
    image_metrics = []
    orig = original_images[i]

    for j in range(num_settings):
        # Inverse
        s_inv = ssim(orig, restored_images_inverse[i][j], data_range=1.0)
        p_inv = psnr(orig, restored_images_inverse[i][j], data_range=1.0)

        # Wiener
        s_wie = ssim(orig, restored_images_wiener[i][j], data_range=1.0)
        p_wie = psnr(orig, restored_images_wiener[i][j], data_range=1.0)

        # CLSF
        s_clsf = ssim(orig, restored_images_clsf[i][j], data_range=1.0)
        p_clsf = psnr(orig, restored_images_clsf[i][j], data_range=1.0)

        image_metrics.append({
            'Setting': f"Set {j} (a={a_params[j]}, b={b_params[j]}, T={T_params[j]})",
            'SSIM Inverse': s_inv, 'PSNR Inverse': p_inv,
            'SSIM Wiener': s_wie, 'PSNR Wiener': p_wie,
            'SSIM CLSF': s_clsf, 'PSNR CLSF': p_clsf
        })

    df_current = pd.DataFrame(image_metrics)

    title = f"Metrics for Image: {image_names[i]}"
    styled_tables.append(style_metrics_table(df_current, title))

for table in styled_tables:
    display(table)

## 4) Frequency analysis
In this section, we analyze the frequency content of the degraded images using the Fast Fourier Transform (FFT). (Just for the astronaut image)

In [ ]:
# Getting the FFTs
ffts_list = [
    [np.fft.fft2(original_images[0])],                          # Original
    [np.fft.fft2(img) for img in degraded_images[0]],           # Degraded
    [np.fft.fft2(img) for img in restored_images_inverse[0]],   # Inverse
    [np.fft.fft2(img) for img in restored_images_wiener[0]],    # Wiener
    [np.fft.fft2(img) for img in restored_images_clsf[0]]       # CLSF
]

def get_log_spectrum(img_fft):
    return np.log(1 + np.abs(np.fft.fftshift(img_fft)))

# Plotting the FFTs
fig, axes = plt.subplots(num_settings, 5, figsize=(22, 18))
fig.suptitle('Frequency Domain Analysis: Astronaut Image (Log-Magnitude Spectrum)',
             fontsize=24, fontweight='bold', y=1.02)

col_titles = ['Original', 'Degraded', 'Inverse Filter', 'Wiener Filter', 'CLSF Filter']

for i in range(num_settings):
    for j in range(5):
        data_to_plot = ffts_list[j][0] if j == 0 else ffts_list[j][i]

        ax = axes[i, j]
        spectrum = get_log_spectrum(data_to_plot)

        im = ax.imshow(spectrum, cmap='magma')

        if i == 0: ax.set_title(col_titles[j], fontweight='bold', fontsize=14)

        if j == 0:
            n_var = noise_variance[i] if isinstance(noise_variance, (list, np.ndarray)) else noise_variance

            params_text = f"Setting {i}\na={a_params[i]:.2f}\n b={b_params[i]:.2f}\nT={T_params[i]:.2f}\nN_v: {n_var:.4f}"
            ax.set_ylabel(params_text, fontweight='bold', fontsize=12, rotation=0, labelpad=40, va='center')

        if j == 2: ax.set_xlabel(f"Threshold={inverse_threshold[j]:.2f}", fontweight='bold', fontsize=12)
        if j == 3: ax.set_xlabel(f"K={wiener_K[j]:.2f}", fontweight='bold', fontsize=12)
        if j == 4: ax.set_xlabel(f"gamma={gammas_c[0][j]:.2f}", fontweight='bold', fontsize=12)

        ax.set_xticks([])
        ax.set_yticks([])

plt.tight_layout()
plt.show()